# RAG on Raw Data (No Entity Resolution)

This notebook builds a RAG chatbot directly on the **raw** Open Sanctions and Open Ownership datasets — no Senzing entity resolution involved. The goal is to show what RAG looks like *before* entity resolution so we can compare it with the entity-resolved version in notebook 06.

**Key differences from notebook 06:**
- Vectorizes the **original source records** (340 records), not resolved entities (196)
- Creates human-readable text from each JSON record instead of vectorizing raw JSON
- Supports both **Anthropic** and **OpenAI** as the LLM backend

In [1]:
import json
import os

import anthropic
import lancedb
import openai
import pyarrow as pa
from sentence_transformers import SentenceTransformer

print("All imports successful")

All imports successful


## Clear LanceDB

Completely remove all existing LanceDB data so we start fresh with only the raw source records.

In [2]:
LANCEDB_PATH = "/workspace/lancedb_data"


def destroy_lancedb(path: str) -> None:
    """Completely clear all LanceDB data by dropping every table."""
    db = lancedb.connect(path)
    table_names = db.table_names()
    if table_names:
        for name in table_names:
            db.drop_table(name)
            print(f"Dropped table: {name}")
        print(f"Cleared {len(table_names)} table(s) from LanceDB at: {path}")
    else:
        print(f"No tables found in LanceDB at: {path}")


destroy_lancedb(LANCEDB_PATH)

No tables found in LanceDB at: /workspace/lancedb_data


/tmp/ipykernel_2065/3972754603.py:7: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  table_names = db.table_names()


## Load Raw Data

Load the original Open Sanctions and Open Ownership JSON files (JSONL format — one JSON object per line).

In [3]:
DATA_DIR = "/workspace/data"


def load_jsonl(filepath: str) -> list[dict]:
    """Load a JSONL file (one JSON object per line)."""
    records = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


sanctions_records = load_jsonl(os.path.join(DATA_DIR, "open-sanctions-v4.jsonl"))
ownership_records = load_jsonl(os.path.join(DATA_DIR, "open-ownership-v4.jsonl"))

print(f"Open Sanctions records: {len(sanctions_records)}")
print(f"Open Ownership records: {len(ownership_records)}")
print(f"Total raw records: {len(sanctions_records) + len(ownership_records)}")

Open Sanctions records: 24
Open Ownership records: 316
Total raw records: 340


## Build Meaningful Text from Raw Records

Instead of vectorizing the raw JSON, we create a human-readable text description for each record. This gives the embedding model meaningful semantic content to work with.

In [4]:
def _get_features(record: dict) -> list[dict]:
    """Return the FEATURES list, falling back to an empty list."""
    return record.get("FEATURES", [])


def get_name(record: dict) -> str:
    """Extract the primary name from a record (v4 FEATURES format)."""
    for feat in _get_features(record):
        for key in ("NAME_FULL", "NAME_ORG", "PRIMARY_NAME_ORG", "PRIMARY_NAME_FULL"):
            if feat.get(key):
                return feat[key]
    # Fallback to top-level fields
    if record.get("PRIMARY_NAME_FULL"):
        return record["PRIMARY_NAME_FULL"]
    return "Unknown"


def record_to_text(record: dict) -> str:
    """Convert a raw JSON record into a meaningful human-readable string."""
    parts = []
    features = _get_features(record)

    name = get_name(record)

    # Extract fields from FEATURES
    record_type = "UNKNOWN"
    addresses = []
    dob = None
    nationality = None
    reg_country = None
    reg_date = None
    identifiers = []
    relationships = []

    for feat in features:
        if feat.get("RECORD_TYPE"):
            record_type = feat["RECORD_TYPE"]
        if feat.get("ADDR_FULL"):
            addresses.append(feat["ADDR_FULL"])
        if feat.get("DATE_OF_BIRTH"):
            dob = feat["DATE_OF_BIRTH"]
        if feat.get("NATIONALITY"):
            nationality = feat["NATIONALITY"]
        if feat.get("REGISTRATION_COUNTRY"):
            reg_country = feat["REGISTRATION_COUNTRY"]
        if feat.get("REGISTRATION_DATE"):
            reg_date = feat["REGISTRATION_DATE"]
        # Identifiers
        id_type = feat.get("OTHER_ID_TYPE") or feat.get("NATIONAL_ID_TYPE") or ""
        id_num = feat.get("OTHER_ID_NUMBER") or feat.get("NATIONAL_ID_NUMBER") or ""
        if id_type and id_num:
            identifiers.append(f"{id_type}: {id_num}")
        elif id_num:
            identifiers.append(id_num)
        # Relationships
        if feat.get("REL_POINTER_ROLE"):
            role = feat["REL_POINTER_ROLE"]
            domain = feat.get("REL_POINTER_DOMAIN", "")
            key = feat.get("REL_POINTER_KEY", "")
            from_date = feat.get("REL_POINTER_FROM_DATE", "")
            thru_date = feat.get("REL_POINTER_THRU_DATE", "")
            rel_str = f"{role} (ref: {domain}/{key})"
            if from_date:
                rel_str += f" from {from_date}"
            if thru_date:
                rel_str += f" to {thru_date}"
            relationships.append(rel_str)

    source = record.get("DATA_SOURCE", "UNKNOWN")
    parts.append(f"{name} is a {record_type.lower()} from the {source} dataset.")

    if addresses:
        parts.append(f"Address: {'; '.join(addresses)}.")
    if dob:
        parts.append(f"Date of birth: {dob}.")
    if reg_date:
        parts.append(f"Registration date: {reg_date}.")
    # Top-level dates (Open Ownership)
    if record.get("REGISTRATION_DATE"):
        parts.append(f"Registration date: {record['REGISTRATION_DATE']}.")
    if record.get("dissolutionDate"):
        parts.append(f"Dissolution date: {record['dissolutionDate']}.")
    if nationality:
        parts.append(f"Nationality: {nationality.upper()}.")
    if reg_country:
        parts.append(f"Registered in: {reg_country.upper()}.")

    # Risks (top-level fields in v4)
    risk_topics = record.get("risk_topics", "")
    if risk_topics:
        parts.append(f"Risk flags: {risk_topics}.")

    if identifiers:
        parts.append(f"Identifiers: {', '.join(identifiers)}.")
    if relationships:
        parts.append(f"Relationships: {'; '.join(relationships)}.")

    return " ".join(parts)


# Test with a few records
print("=== Sample Open Sanctions record ===")
print(record_to_text(sanctions_records[0]))
print()
print("=== Sample Open Ownership record (org) ===")
print(record_to_text(ownership_records[0]))
print()
# Find a person record in ownership
person_oo = next(
    (r for r in ownership_records
     if any(f.get("RECORD_TYPE") == "PERSON" for f in r.get("FEATURES", []))),
    None
)
if person_oo:
    print("=== Sample Open Ownership record (person) ===")
    print(record_to_text(person_oo))

=== Sample Open Sanctions record ===
Abassin BADSHAH is a person from the OPEN-SANCTIONS dataset. Address: 31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL. Date of birth: 1985-05-12. Nationality: GB. Risk flags: corp.disqual. Identifiers: OPEN-SANCTIONS: NK-25vyVFzt8vdJGgAXMRTwTJ. Relationships: Directorship (ref: OPEN-SANCTIONS/NK-SKAADAiqiZ78JsJjeg72Te); Directorship (ref: OPEN-SANCTIONS/NK-3p3mmVWmjwVtTfKchz4kNE).

=== Sample Open Ownership record (org) ===
GOLD WYNN UK HOLDINGS LIMITED is a organization from the OPEN-OWNERSHIP dataset. Address: C/O Fladgate Llp, 16 Great Queen Street, London, WC2B 5DG. Registration date: 2020-03-18. Registered in: GB. Identifiers: GB-COH: 12524623. Relationships: shareholding 75% 100% (ref: OOR/7584591804488095167); voting_rights 75% 100% (ref: OOR/7584591804488095167); appointment_of_board (ref: OOR/7584591804488095167).

=== Sample Open Ownership record (person) ===
Kenneth Kurt Hansen is a person from the OPEN-OWNERSHIP dataset. Addre

## Vectorize and Store in LanceDB

Embed all text descriptions using the same `all-MiniLM-L6-v2` model, then store them alongside metadata in a fresh LanceDB table.

In [5]:
print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding model loaded (dimension: {embedding_model.get_sentence_embedding_dimension()})")

# Combine all records and build text + metadata
all_records = sanctions_records + ownership_records
print(f"\nProcessing {len(all_records)} records...")

rows = []
seen = set()
for record in all_records:
    # Deduplicate by (DATA_SOURCE, RECORD_ID)
    key = (record.get("DATA_SOURCE", ""), record.get("RECORD_ID", ""))
    if key in seen:
        continue
    seen.add(key)

    text = record_to_text(record)
    name = get_name(record)
    # Extract RECORD_TYPE from FEATURES list
    record_type = ""
    for feat in record.get("FEATURES", []):
        if feat.get("RECORD_TYPE"):
            record_type = feat["RECORD_TYPE"]
            break
    rows.append({
        "record_id": record.get("RECORD_ID", ""),
        "data_source": record.get("DATA_SOURCE", ""),
        "record_type": record_type,
        "name": name,
        "text": text,
    })

print(f"Unique records to vectorize: {len(rows)}")

# Batch encode all texts
texts = [r["text"] for r in rows]
embeddings = embedding_model.encode(texts, show_progress_bar=True)

for i, row in enumerate(rows):
    row["vector"] = embeddings[i].tolist()

print(f"Embeddings created: {len(embeddings)} x {embeddings.shape[1]}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded (dimension: 384)

Processing 340 records...
Unique records to vectorize: 282


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embeddings created: 282 x 384


In [6]:
# Store in LanceDB
db = lancedb.connect(LANCEDB_PATH)
table = db.create_table("raw_records", data=rows)

print(f"LanceDB table 'raw_records' created with {table.count_rows()} rows")
print(f"\nSample entries:")
sample = table.to_pandas().head(3)
print(sample[["record_id", "data_source", "record_type", "name"]].to_string())

LanceDB table 'raw_records' created with 282 rows

Sample entries:
                   record_id     data_source   record_type                     name
0  NK-25vyVFzt8vdJGgAXMRTwTJ  OPEN-SANCTIONS        PERSON          Abassin BADSHAH
1  NK-3p3mmVWmjwVtTfKchz4kNE  OPEN-SANCTIONS  ORGANIZATION            LMAR (GB) LTD
2  NK-auyPsLrBzRoxjCRWgjBvas  OPEN-SANCTIONS  ORGANIZATION  WANDLE HOLDINGS LIMITED


## Set Up LLM Clients

In [7]:
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Anthropic client ready:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("OpenAI client ready:", bool(os.getenv("OPENAI_API_KEY")))

Anthropic client ready: True
OpenAI client ready: True


## RAG Query Function

Vector search over raw records, then query either Anthropic (Claude) or OpenAI (GPT) to answer the question. The `provider` parameter controls which LLM backend is used.

In [8]:
SYSTEM_PROMPT = (
    "Answer questions about a corporate ownership and sanctions dataset. "
    "You have access to individual raw source records from Open Sanctions and "
    "Open Ownership. These records have NOT been entity-resolved — the same "
    "real-world entity may appear as separate records with slightly different "
    "names or details. You do not have relationship or graph data beyond what "
    "is mentioned in individual records."
)


def ask_raw_rag(question: str, provider: str = "anthropic", top_k: int = 10) -> None:
    """
    RAG over raw (non-entity-resolved) records.

    Parameters
    ----------
    question : str
        The user's question.
    provider : str
        LLM backend — "anthropic" (Claude) or "openai" (GPT).
    top_k : int
        Number of records to retrieve.
    """
    if provider not in ("anthropic", "openai"):
        raise ValueError(f"provider must be 'anthropic' or 'openai', got '{provider}'")

    print(f"\nQuestion: {question}")
    print(f"Provider: {provider}")
    print("=" * 70)

    # Step 1: Vector search
    question_embedding = embedding_model.encode(question).tolist()
    results = table.search(question_embedding).limit(top_k).to_list()
    print(f"Retrieved {len(results)} records")

    # Step 2: Build context
    context_parts = ["RAW SOURCE RECORDS:\n"]
    for r in results:
        context_parts.append(f"- [{r['data_source']}] {r['text']}")
        context_parts.append("")
    context = "\n".join(context_parts)

    # Step 3: Query LLM
    print("Querying LLM...")
    user_message = f"Context:\n{context}\n\nQuestion: {question}"

    if provider == "anthropic":
        response = anthropic_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}],
        )
        answer = response.content[0].text
    else:
        response = openai_client.chat.completions.create(
            model="gpt-5.4-nano",
            max_tokens=2048,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ],
        )
        answer = response.choices[0].message.content

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)
    print(answer)
    print("=" * 70)

## Interactive Query Session

Ask questions against the raw (non-entity-resolved) data. Change `provider` below to `"openai"` to use GPT instead of Claude.

In [9]:
provider = "anthropic"  # change to "openai" to use GPT-5.4 nano

print("RAG on Raw Data (No Entity Resolution) - Interactive Session")
print("=" * 70)
print("Ask questions about the corporate ownership and sanctions data.")
print("These are RAW records — no entity resolution has been applied.")
print(f"Current LLM provider: {provider}")
print("Type 'quit' to exit.\n")

while True:
    question = input("Your question: ").strip()

    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break

    if not question:
        continue

    try:
        ask_raw_rag(question, provider=provider)
        print()
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

RAG on Raw Data (No Entity Resolution) - Interactive Session
Ask questions about the corporate ownership and sanctions data.
These are RAW records — no entity resolution has been applied.
Current LLM provider: anthropic
Type 'quit' to exit.



Your question:  tell me about said kerimov



Question: tell me about said kerimov
Provider: anthropic
Retrieved 10 records
Querying LLM...

ANSWER
# Said Kerimov

Based on the available records, here's what we know about Said Kerimov:

## Basic Information
- **Date of Birth:** July 1995 (records show both July 1 and July 6, 1995)
- **Nationality:** Russian (RU)

## Addresses
Said Kerimov is associated with multiple addresses across London and Moscow:

**London:**
- 16 Berkeley Street, London, W1J 8DZ
- 8th Floor, 20 Farringdon Street, London, EC4A 4AB

**Moscow:**
- Apt 270, Build. 31, Pyatnitskoe Shosse, Moscow, 123430

## Risk Profile
According to the Open Sanctions dataset, Said Kerimov has significant risk flags:
- **Sanctioned individual**
- **RCA (Relative or Close Associate)** designation

## Family Connection
The records indicate a **family relationship with Suleyman Abusaidovich Kerimov** (Q447250), a prominent Russian oligarch and politician born in 1966. This family connection appears to be the basis for Said's "RCA" 

Your question:  tell me about abassin badshah



Question: tell me about abassin badshah
Provider: anthropic
Retrieved 10 records
Querying LLM...

ANSWER
# Abassin Badshah

Based on the available records, here's what I can tell you about Abassin Badshah:

## Basic Information
- **Date of Birth:** May 1985 (appears as May 1, 1985 in some records and May 12, 1985 in others)
- **Nationality:** British (GB)

## Addresses
Abassin Badshah is associated with two addresses in Bromley, UK:
1. **31 Quernmore Close, Bromley, BR1 4EL** (also listed as "31 Quernmore Close, Bromley, Kent")
2. **3 Market Parade, 41 East Street, Bromley, BR1 1QN**

## Risk Flags
⚠️ **Important:** One record shows Abassin Badshah has a risk flag for **"corp.disqual"** - this typically indicates corporate disqualification, meaning this person may have been disqualified from serving as a company director.

## Business Relationships
The records indicate Abassin Badshah has held directorship positions (references to two directorships are mentioned in the Open Sanctions 

Your question:  q


Goodbye!
